In [ ]:
import math
import numpy
import sys
import os
from pathlib import Path
local_path = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(local_path))
import dolfin_mech as dmech
from fenics import *
import dolfin
from dolfin_mech.Problem_Hyperelasticity_MicroPoroFlow import MicroPoroFlowHyperelasticityProblem
from dolfin_mech.run_MicroPoroflow import run_MicroPoroflow
import myPythonLibrary as mypy

In [ ]:

# ----------------- Run with Options -----------------

mat_params = {
    "alpha":0.16,
    "gamma":0.5,
    "c1":0.2,
    "c2":0.4,
    "kappa":1,
    "eta":1e-5}


res_folder = sys.argv[0][:-3]
test = mypy.Test(
    res_folder=res_folder,
    perform_tests=0,
    stop_at_failure=1,
    clean_after_tests=0,
    tester_numpy_tolerance=1e-2)

dim_lst  = [ ]
dim_lst += [2]
# dim_lst += [3]
Ex_values = [0.0, 0.1, 0.2]
pf_values = [0.0, 0.03,0.06]
#Ex_values = [0.0, 0.1, 0.2] 
#pf_values = [0]
p_bar_lst = [0.0, 0.1, 0.1]#[0.0, 0.1, 0.2]

pl_bar_ini_lst = [0.0, 0.0]
pl_bar_fin_lst = [0.0, 0.0]

grad_p_bar_x_ini_lst = [0.0, 0.001]
grad_p_bar_x_fin_lst = [0.001, 0.001]

grad_p_bar_y_ini_lst = [0.0, 0.001]
grad_p_bar_y_fin_lst = [0.001, 0.001]

Theta_in_lst = [0.0,0]   
Theta_out_lst = [0.0,0]


flow_loading_params = {
    # pressure (scalar)
    "pl_bar_ini_lst": pl_bar_ini_lst,
    "pl_bar_fin_lst": pl_bar_fin_lst,

    # pressure gradient (2D)
    "grad_p_bar_x_ini_lst": grad_p_bar_x_ini_lst,
    "grad_p_bar_x_fin_lst": grad_p_bar_x_fin_lst,
    "grad_p_bar_y_ini_lst": grad_p_bar_y_ini_lst,
    "grad_p_bar_y_fin_lst": grad_p_bar_y_fin_lst,

    # Theta (keep simple: scalar ini/fin per step)
    "Theta_in_ini_lst":  [0.0, 0.0],  
    "Theta_in_fin_lst":  Theta_in_lst, 
    "Theta_out_ini_lst": [0.0, 0.0],
    "Theta_out_fin_lst": Theta_out_lst,
}
for dim in dim_lst:

    bcs_lst  = [      ]
    #bcs_lst += ["kubc"]
    bcs_lst += ["pbc" ]
    for bcs in bcs_lst:

        load_lst  = [                     ]
        #load_lst += ["K_vs_U"]
        load_lst += ["K_vs_pf"]
        for load in load_lst:

            if load == "K_vs_U":
                load_params = {}
                def set_sigma_bar_all_zero(except00=False):
                    for i in range(dim):
                        for j in range(dim):
                            if except00 and (i == 0 and j == 0):
                                continue
                            load_params[f"sigma_bar_{i}{j}"] = 0.0

                for pf in pf_values:

                    print("dim =",dim)
                    print("bcs =",bcs)
                    print("load =",load)
                    print("pf   =",pf)


                    #res_basename  = sys.argv[0][:-3]
                    res_basename = "-dim="+str(dim)
                    res_basename += "-bcs="+str(bcs)
                    res_basename += "-load="+str(load)
                    res_basename += "-pf="+str(pf)

                    load_params["pf_lst"] = [pf,pf]

                    load_params["U_bar_00_lst"] = [0,0.3]
                    for i in range(dim):
                        for j in range(dim):
                            if ((i != 0) or (j != 0)):
                                load_params["sigma_bar_"+str(i)+str(j)] = 0.

                    # load_params["U_bar_00_lst"] = [0, 0.3]
                    # load_params["U_bar_11_lst"] = [0, 0.3]

                    # for i in range(dim):
                    #     for j in range(dim):
                    #         if (i, j) not in [(0, 0), (1, 1)]:
                    #             load_params["sigma_bar_"+str(i)+str(j)] = 0.

                    # for i in range(dim):
                    #     for j in range(dim):
                    #         load_params[f"U_bar_{i}{j}_lst"] = [0.0, 0.0]


                    # for i in range(dim):
                    #     for j in range(dim):
                    #         if ((i != 0) or (j != 0)):
                    #             load_params["sigma_bar_"+str(i)+str(j)] = 0.

                    run_MicroPoroFlowHyperelasticity(
                        dim=dim,
                        mesh_params={"dim":dim, "xmin":0., "ymin":0., "zmin":0., "xmax":1., "ymax":1., "zmax":1., "xshift":-0.5, "yshift":-0.5, "zshift":-0.5, "r0":0.2, "l":0.05, "mesh_filebasename":res_folder+"/"+"mesh"},
                        mat_params={
                                "skel": {"parameters": mat_params, "scaling": "no"},
                                "bulk": {"parameters": mat_params, "scaling": "no"},
                                "pore": {"parameters": mat_params, "scaling": "no"}
                            },
                        flow_params={ 
                            "k_l": dolfin.Constant(((1e-6, 0.0),
                                (0.0, 1e-6))),
                            "use_kozeny_carman": False
                            },
                        flow_loading_params=flow_loading_params,
                        porosity_params={
                            "type": "constant",  # can be "constant", "function_constant", or "random"
                            "val": 0.3
                        },  
                        
                        bcs=bcs,
                        step_params = {
                            "n_steps": 2,
                            "Deltat_lst": [1e-2, 1e-1],     
                            "dt_ini_lst": [2e-3, 1e-3],     
                            "dt_min_lst": [2e-3, 1e-4],     
                            "dt_max_lst": [2e-3, 5e-3],     
                        },
                        load_params=load_params,
                        res_basename=res_folder+"/"+res_basename,
                        verbose=0)

                    test.test(res_basename)


            if load == "K_vs_pf":
                for Ex in Ex_values:

                    load_params = {}
                    load_params["U_bar_00_lst"] = [Ex, Ex]      
                    pf_target = 0.2
                    load_params["pf_lst"] = [0.0, pf_target]     

                    for i in range(dim):
                        for j in range(dim):
                            if (i, j) != (0, 0):
                                load_params[f"sigma_bar_{i}{j}"] = 0.0

                    res_basename  = f"-dim={dim}-bcs={bcs}-load=K_vs_pf-Ex={Ex}"

                    run_MicroPoroFlowHyperelasticity(
                        dim=dim,
                        mesh_params={"dim":dim, "xmin":0., "ymin":0., "zmin":0., "xmax":1., "ymax":1., "zmax":1., "xshift":-0.5, "yshift":-0.5, "zshift":-0.5, "r0":0.2, "l":0.05, "mesh_filebasename":res_folder+"/"+"mesh"},
                        mat_params={
                                "skel": {"parameters": mat_params, "scaling": "no"},
                                "bulk": {"parameters": mat_params, "scaling": "no"},
                                "pore": {"parameters": mat_params, "scaling": "no"}
                            },
                        flow_params={ 
                            "k_l": dolfin.Constant(((1e-6, 0.0),
                                (0.0, 1e-6))),
                            "use_kozeny_carman": False,
                            },
                        flow_loading_params=flow_loading_params,
                        porosity_params={
                            "type": "constant",  # can be "constant", "function_constant", or "random"
                            "val": 0.3
                        },  
                        
                        bcs=bcs,
                        step_params = {
                            "n_steps": 2,
                            "Deltat_lst": [1e-2, 1e-1],     
                            "dt_ini_lst": [5e-3, 1e-3],     
                            "dt_min_lst": [5e-3, 1e-4],     
                            "dt_max_lst": [5e-3, 5e-3],     
                        },
                        load_params=load_params,
                        res_basename=res_folder+"/"+res_basename,
                        verbose=0)
                    test.test(res_basename)

                
